# Simple baseline (XGBoost)

This notebook tries to use XGBoost to improve our previous baseline of 61%.

In [9]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

import duckdb # we rely on duckdb for parsing movedata
duckdb.sql("LOAD aixchess")

In [3]:
# It will go like:
# model = XGBClassifier()
# model.fit(X_train, y_train)
# model.score(X_test, y_test)

In [4]:
data_path = "subset.parquet"
df = pd.read_parquet(data_path)
df.head()

,lichess_id,tournament,movedata,clocks_white,clocks_black,evals,ply_count,white,black,white_rating,...,time_increment,result,termination,white_rating_diff,black_rating_diff,eco,opening,white_title,black_title,utc_timestamp
0,eh5dKZuD,NaN,b'\xf1\x12\xef\xae\x9c\x86\xe3\x01\xe50\xfe\x9...,"[180, 179, 177, 176, 175, 171, 165, 162, 158, ...","[180, 179, 178, 175, 173, 163, 154, 144, 140, ...",None,29,Panchito0O,PauloPeru78,1247,...,0.0,1-0,Normal,5.0,-5.0,C25,Vienna Game: Anderssen Defense,NaN,NaN,2025-01-01 00:00:05
1,6yXyLl2M,NaN,b'e\x9f\xe3\xbf\xa9\xdc>\x8c\xea[G\x00_k:\xf5)...,"[180, 179, 178, 177, 174, 173, 172, 171, 169, ...","[180, 180, 179, 178, 178, 177, 176, 176, 174, ...",None,104,igloknight,atacan3131,1577,...,0.0,0-1,Time forfeit,-6.0,6.0,B12,Caro-Kann Defense: Masi Variation,NaN,NaN,2025-01-01 00:00:05
2,SZW2amZz,NaN,"b""-\xb4K\xb2C\xdf\xa3Wem\x88(\xfe'|\x93\xa2Q`\...","[180, 179, 179, 177, 175, 173, 172, 171, 169, ...","[180, 178, 175, 174, 173, 171, 169, 168, 163, ...",None,87,draughts2chess,xhoxhi64,1043,...,0.0,1-0,Normal,10.0,-5.0,D00,Queen's Pawn Game,NaN,NaN,2025-01-01 00:00:05
3,VGhr7swl,NaN,b'\xb3s\xaff\x0b\x8c|\x94\xf7\xcb\xbb\xa7\xb22...,"[180, 176, 174, 172, 168, 167, 165, 158, 142, ...","[180, 180, 180, 179, 177, 176, 176, 166, 158, ...",None,58,GodofPastries,Mickey187,2015,...,0.0,1-0,Normal,6.0,-11.0,B13,Caro-Kann Defense: Exchange Variation,NaN,NaN,2025-01-01 00:00:05
4,aQTAJkjw,NaN,b'\xb3\xc9\xf7\xbf\xf4\x11\xde\x9e\x85*\x97\xc...,"[180, 179, 179, 177, 177, 177, 176, 176, 176, ...","[180, 178, 177, 177, 177, 175, 173, 172, 171, ...",None,112,elprimo27,knocikIII,2139,...,0.0,0-1,Normal,-6.0,5.0,B10,Caro-Kann Defense: Endgame Variation,NaN,NaN,2025-01-01 00:00:05


In [5]:
games = duckdb.read_parquet("subset.parquet")

game_positions = duckdb.sql("""
SELECT
  lc.* EXCLUDE (movedata, clocks_white, clocks_black, tournament),                -- consider listing columns instead of *
  t.ply,
  UNNEST(board_at_position(lc.movedata, t.ply))
FROM games AS lc
CROSS JOIN LATERAL (
  SELECT 1 + CAST(floor(random() * lc.ply_count) AS INTEGER) AS ply
) AS t
""")
df = game_positions.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [24]:
result_map = {'1-0': 0, '0-1': 1, '1/2-1/2': 2}

def preprocess_df(df):
    # Select out the features and labels
    labels = 'result'
    
    categorical_features = [chr(file+ord('a'))+str(rank+1) for rank in range(8) for file in range(8)]
    features = ['white_rating', 'black_rating', 'ply']
    
    one_hot_df = pd.DataFrame({f'{col}_{piece}': df[col] == piece for piece in list('prnbqkPRNBQK') for col in categorical_features})
    X_df = pd.concat([df[features], one_hot_df], axis='columns')
    X_df['ply'] = (X_df['ply'] % 2 == 0)
    
    # XGBoost decision trees do not require scaling
    # X_df['white_rating'] = (X_df['white_rating'] - 1500) / 400 
    # X_df['black_rating'] = (X_df['black_rating'] - 1500) / 400

    y_df = df[labels].map(result_map)
    return X_df, y_df

# Note: this function has changed from the previous notebook
def preprocess(X_df, y_df, test_size=0.2, random_state=None):
    X, y = X_df.to_numpy(), y_df.to_numpy()
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

def train_baseline(X_train, y_train) -> XGBClassifier:
    model = XGBClassifier()
    model.fit(X_train, y_train)
    return model

def eval_model(model, X_test, y_test):
    y_preds = model.predict(X_test)
    num_correct = (y_preds == y_test).sum()
    total = y_test.shape[0]
    print(f"Accuracy: {num_correct}/{total} = {(num_correct / total):.3f}")
    return dict(num_correct=num_correct, total=total, accuracy=num_correct / total)



In [25]:
X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train)
_ = eval_model(model, X_test, y_test)

Accuracy: 12780/20000 = 0.639


### Accuracy jump, weight visualization

We've improved accuracy again just by adding piece positions (61% -> 64%). So, this means that our model is benefitting from being aware of non-linear interactions between pieces. Let's try adding some features and tuning and see if we can't squeeze out a couple of more points.

In [30]:
def train_baseline(X_train, y_train, **kwargs):
    model = XGBClassifier(**kwargs)
    model.fit(X_train, y_train)
    return model

X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=500, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 12939/20000 = 0.647


In [31]:
# Minor improvement. Let's see about overfitting.
_ = eval_model(model, X_train, y_train)

Accuracy: 61827/80000 = 0.773


In [34]:
# Clearly, we are starting to overfit. More data will likely help.
df.shape[0]

100000

In [44]:
# Let's scale up the data 10x.

games = duckdb.read_parquet("games.parquet")
subset = duckdb.sql("FROM games WHERE result != '*' LIMIT 1000000")
game_positions = duckdb.sql("""
SELECT
  lc.* EXCLUDE (movedata, clocks_white, clocks_black, tournament),
  t.ply,
  UNNEST(board_at_position(lc.movedata, t.ply))
FROM subset AS lc
CROSS JOIN LATERAL (
  SELECT 1 + CAST(floor(random() * lc.ply_count) AS INTEGER) AS ply
) AS t ORDER BY random()
""")
df = game_positions.df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [45]:
X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=500, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 128524/200000 = 0.643


In [46]:
_ = eval_model(model, X_train, y_train)

Accuracy: 543352/800000 = 0.679


In [47]:
# Let's try scaling up even more.
X_df, y_df = preprocess_df(df)
X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=1000, max_depth=8, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 129492/200000 = 0.647


In [48]:
_ = eval_model(model, X_train, y_train)

Accuracy: 568614/800000 = 0.711


In [49]:
# Let's try scaling up even more.
# X_df, y_df = preprocess_df(df)
# X_train, X_test, y_train, y_test = preprocess(X_df, y_df, random_state=0)
model = train_baseline(X_train, y_train, n_estimators=1000, max_depth=10, learning_rate=0.1)
_ = eval_model(model, X_test, y_test)

Accuracy: 130116/200000 = 0.651


### Conclusion

We've improved accuracy again by capturing non-linearities via XGBoost. This has resulted in a test accuracy improvement of between 3-4%. The only problem
is that we are still overfitting -- even after scaling up to a million positions.

A few directions we could try:
1. Scaling up data and the model further. Given that we barely got an improvement of 1% after scaling data 10x, it's likely we are beginning to hit diminishing
returns.
2. Add time-control / clock features: given blunder rates in fast time controls, it's likely that this would improve model prediction a non-trivial amount.
3. Try a different model architecture: Chess is fundamentally a spatial game -- it's possible that a deep neural network such as a CNN on a spatial board could
improve performance.

The next notebook will explore a CNN, because clock info is almost guaranteed to produce an improvement, whereas deep networks are yet untested.